Script to **create synthetic model** (defined in the settings file)

All rights reserved. Material from GEOFCAN 2025 Workshop led by Coline Mollaret

In [ ]:
#############################################
import sys, os
from pathlib import Path
# try and except functions to change the directory only once to stay in the right folder even the script/cell is run again
try:  findpaths
except NameError:  
    os.chdir(os.path.abspath("..")) # to find "fpinv" in the main folder
    findpaths=True
print(os.getcwd()) #print the current working directory
#############################################

import numpy as np
import pygimli as pg
import pygimli.meshtools as mt
import matplotlib.pyplot as plt
from sys import path
from fpinv import FourPhaseModel, set_style

import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR) # to get rid of "matplotlib.font_manager - WARNING - findfont: Font family 'Roboto' not found."


##### Creates new directories if not existing yet
Path(os.getcwd()+'/results/syn/').mkdir(parents=True, exist_ok=True)
Path(os.getcwd()+'/results/mesh/').mkdir(parents=True, exist_ok=True)

In [ ]:
### SYNTHETIC CASE ##########################################
case = '3L'
#case = '3L+block'

petromodel ='A'

fs=10
set_style(fs, style="default")
 
# Import profile specific settings if they exist, otherwise import default settings:
if os.path.exists(os.path.abspath('profile_settings/'+case+'_settings.py')):
    exec(open('profile_settings/'+case+'_settings.py', 'r').read()) # reads case specific settings
    print('case settings exist and were loaded')
else:
    exec(open('profile_settings/test_2L_5m_default_settings.py', 'r').read()) # reads case specific settings
    print('case settings do not exist, default settings are used')

geom.save('data/'+case+'_geom.bms')

# UNSTRUCTURED MESH 
mesh = mt.createMesh(geom, area=area)
pg.show(mesh, markers=True, showMesh=True)

def to_mesh(data):
    return data[mesh.cellMarkers()]

In [ ]:
### CREATE SYNTHETIC MODEL ####################################################
# Model creation based on prescribed pore fractions
if petromodel=="A":
    fpm = FourPhaseModel(vw=vw, vi=vi, va=va, vr=vr, phi=philayers, n=n, m=m, a=a, rhow=rhow)
elif petromodel=="G":
    fpm = FourPhaseModelSomerton(phi=philayers, rhow=rhow, rhoi=rhoi, rhoa=rhoa, rhor=rhor)
print(falayers + filayers + fwlayers + frlayers)
rholayers = fpm.rho(fwlayers, filayers, falayers, frlayers)
vellayers = 1. / fpm.slowness(fwlayers, filayers, falayers, frlayers)

rholayers = np.array(rholayers)
vellayers = np.array(vellayers)
rhotrue = to_mesh(rholayers)
veltrue = to_mesh(vellayers)

# Save true model and mesh
fa = to_mesh(falayers)
fi = to_mesh(filayers)
fw = to_mesh(fwlayers)
fr = to_mesh(frlayers)

fpm.fr = fr
fpm.phi = 1 - fr
#fpm.show(mesh, rhotrue, veltrue)

assert np.allclose(fa + fi + fw + fr, 1)
# np.savez(f'{case}/{case}_true_model.npz', rho=rhotrue, vel=veltrue, fa=fa, fi=fi, fw=fw, fr=fr)
np.savez('data/'+case+'_'+petromodel+'_true_model.npz', rho=rhotrue, vel=veltrue, fa=fa, fi=fi, fw=fw, fr=fr)
    


# Plot SYN RES/VEL (based on prescribed pore fractions)
fig, axes = pg.plt.subplots(2,1,figsize=(10,11))
ax = axes[0]
pg.show(ax=ax, mesh=mesh, data=rhotrue, logScale=True, label='Resistivity [$\Omega$m]', cMin=cMine,
    cMax=cMaxe, grid=True, cMap=cMape)
ax.set_title("SYN RESISTIVITY | based on prescribed PORE FRACTIONS")
# Dynamically display resistivity text annotations for each layer
for i in range(len(rholayers)):
    tr = "%.0f $\Omega$m" % np.rint(rholayers[i])
    ax.text(rm[i][0], rm[i][1], tr, size=11, backgroundcolor='white')
ax.text(mesh.xmin(),mesh.ymax()+2, case, size=15)
ax = axes[1]
pg.show(ax=ax, mesh=mesh, data=veltrue, label='Velocity [m/s]', cMin=cMins, cMax=cMaxs, grid=True, cMap=cMaps)
ax.set_title("SYN VELOCITY | based on prescribed PORE FRACTIONS")
# Dynamically display velocity text annotations for each layer
for i in range(len(vellayers)):
    tv = "%.0f m/s" % np.rint(vellayers[i])
    ax.text(rm[i][0], rm[i][1], tv, size=11, backgroundcolor='white')
pg.plt.savefig('results/syn/'+case+'_'+petromodel+'_SYNmodel_prescr_fractions.png')
    
 

###############################################
print("Number of regions/layers: ", len(rholayers))
print("Layer resistivity (\u03A9m): ", np.rint(rholayers))
print("Layer velocity (m/s)     : ", np.rint(vellayers))

'''
#create matrix with syn_TrueDepthResVel
synd = np.append(worldz,interfaces) #depths
synd = np.append(synd,interfaces)
synd = -np.sort(-synd)
synr = np.repeat(layerres,len(rm)) #resistivities
synv = np.repeat(layervel,len(rm)) #velocities
syn = np.zeros((len(synd),3))
syn[:,0] = np.transpose(synd)
syn[:,1] = np.transpose(synr)
syn[:,2] = np.transpose(synv)
'''

### SAVE SENSORS AND MESH #####################################################
sensors = np.arange(senposmin, senposmax, spacing, dtype="float")
print("Sensor spacing: ", sensors[1]-sensors[0], " m")
print("first/last sensor: ", sensors[0], "/", sensors[len(sensors)-1], " m")

print(len(philayers))

# PRINT PRESCRIBED FRACTIONS

tit="      phi  |   fi   |   fw   |   fa"
print("\n", tit, "\n", "-"*40)
for i in range(len(philayers)):
    layer_str = f'L{i+1}:  {philayers[i]:.2f}  |  {filayers[i]:.2f}  |  {fwlayers[i]:.2f}  |  {falayers[i]:.2f}'
    print(layer_str)
print("\n")

mesh.save('results/mesh/'+case+'_mesh.bms')
sensors.dump('results/mesh/'+case+'_sensors.npy')
shots.dump('results/mesh/'+case+'_shots.npy')
np.savetxt('data/'+case+'_rhotrue.dat', rhotrue)
np.savetxt('data/'+case+'_veltrue.dat', veltrue)


### PLOT PORE FRACTIONS BAR CHART #############################################

fibar=filayers
fwbar=fibar+fwlayers
fabar=fwbar+falayers
frbar=1-philayers

depths=sorted(np.append(worldz,interfaces),reverse=True)
pos=np.arange(len(rm))
print(pos)
fig, ax1 = plt.subplots(figsize=(5,5))
ax1.barh(pos, fibar, color='paleturquoise', edgecolor='k')
lefts=fibar
ax1.barh(pos, fwbar, color='deepskyblue', edgecolor='k', left=lefts)
lefts=fwbar
ax1.barh(pos, fabar, color='azure', edgecolor='k', left=lefts)
lefts=fabar
ax1.barh(pos, frbar, color='grey', edgecolor='k', left=lefts)
plt.xlim([0, 1])
ax1.set_yticks(pos)
ax1.set_xticks(np.arange(0,1.1,0.1))
plt.rc('xtick', labelsize=18) 
#ax1.set_yticklabels(('L1', 'L2', 'L3'),size=18)
layer_labels = [f'L{i+1}' for i in range(len(philayers))]
ax1.set_yticklabels(layer_labels, size=18)
ax1.set_title(f'{case}  |  Fractions per Layer',size=18)
plt.gca().invert_yaxis()
pg.plt.savefig('results/syn/'+case+'_SYNmodel_FractionsPerLayer.png')
